# CFPB Banking Complaints: Tableau Data Preparation

This notebook creates a smaller, Tableau-ready CSV from the cleaned complaint-level export. It keeps one row per complaint so Tableau filters can work across dates, products, companies, issues, states, tags, submission channels, and response outcomes.

The source file is processed in chunks so the full 295.5 MB CSV does not need to be loaded into memory at once.

## Import packages and locate the clean CSV

The code first looks for a CSV containing `clean` in its filename. If it cannot find one, it selects the largest CSV in the project folder. Review the printed path before continuing.

In [ ]:
from pathlib import Path
import pandas as pd

# Folder containing the exported clean CSV and the original project files.
DATA_FOLDER = Path.home() / "Downloads" / "CFPB_Banking_Project"

# Name of the smaller file that this notebook will create.
OUTPUT_FILE = DATA_FOLDER / "cfpb_tableau_data.csv"

# Find CSV files while excluding the output from a previous notebook run.
csv_candidates = [
    path for path in DATA_FOLDER.glob("*.csv")
    if path.name.lower() != OUTPUT_FILE.name.lower()
]

if not csv_candidates:
    raise FileNotFoundError(f"No CSV files were found in: {DATA_FOLDER}")

# Prefer a filename containing the word 'clean'.
clean_candidates = [
    path for path in csv_candidates
    if "clean" in path.stem.lower()
]

# If multiple candidates exist, choose the largest one because the combined
# clean export should be much larger than an individual annual source file.
INPUT_FILE = max(
    clean_candidates if clean_candidates else csv_candidates,
    key=lambda path: path.stat().st_size
)

print(f"Selected input file: {INPUT_FILE}")
print(f"Input size: {INPUT_FILE.stat().st_size / (1024 ** 2):,.1f} MB")
print(f"Output file: {OUTPUT_FILE}")

## Select and verify the dashboard fields

These fields support the planned trend, product, issue, company, geographic, tag, channel, timeliness, and resolution visuals. Redundant timestamps, raw categories, ZIP codes, source filenames, and other unused fields are excluded.

In [2]:
# Keep the columns in this order in the Tableau-ready file.
TABLEAU_COLUMNS = [
    "complaint_id",
    "received_date",
    "received_year",
    "month_start_date",
    "product_group",
    "issue",
    "sub_issue",
    "company",
    "state",
    "tags",
    "submitted_via",
    "company_response",
    "is_timely",
]

# Read only the header so the source columns can be checked quickly.
available_columns = pd.read_csv(INPUT_FILE, nrows=0).columns.tolist()
missing_columns = [
    column for column in TABLEAU_COLUMNS
    if column not in available_columns
]

if missing_columns:
    raise ValueError(
        "The source file is missing required columns: "
        + ", ".join(missing_columns)
    )

print("All 13 required Tableau columns were found.")
print(TABLEAU_COLUMNS)

All 13 required Tableau columns were found.
['complaint_id', 'received_date', 'received_year', 'month_start_date', 'product_group', 'issue', 'sub_issue', 'company', 'state', 'tags', 'submitted_via', 'company_response', 'is_timely']


## Create the Tableau-ready CSV

The source is read in 100,000-row chunks. Dates are standardized as `YYYY-MM-DD`, numeric fields are validated, and missing optional values remain blank. A temporary file is used so an interrupted run does not leave an incomplete final output.

In [ ]:
CHUNK_SIZE = 100_000
TEMP_OUTPUT_FILE = OUTPUT_FILE.with_suffix(".tmp.csv")

# Remove only an unfinished temporary file from an earlier attempt.
if TEMP_OUTPUT_FILE.exists():
    TEMP_OUTPUT_FILE.unlink()

total_rows_written = 0
invalid_received_dates = 0
invalid_month_dates = 0

reader = pd.read_csv(
    INPUT_FILE,
    usecols=TABLEAU_COLUMNS,
    chunksize=CHUNK_SIZE,
    low_memory=False,
)

for chunk_number, chunk in enumerate(reader, start=1):
    # Convert the two dashboard date fields to consistent date strings.
    received_dates = pd.to_datetime(chunk["received_date"], errors="coerce")
    month_dates = pd.to_datetime(chunk["month_start_date"], errors="coerce")

    invalid_received_dates += int(received_dates.isna().sum())
    invalid_month_dates += int(month_dates.isna().sum())

    chunk["received_date"] = received_dates.dt.strftime("%Y-%m-%d")
    chunk["month_start_date"] = month_dates.dt.strftime("%Y-%m-%d")

    # Store identifiers and indicator fields as nullable integers.
    chunk["complaint_id"] = pd.to_numeric(
        chunk["complaint_id"], errors="coerce"
    ).astype("Int64")

    chunk["received_year"] = pd.to_numeric(
        chunk["received_year"], errors="coerce"
    ).astype("Int16")

    chunk["is_timely"] = pd.to_numeric(
        chunk["is_timely"], errors="coerce"
    ).astype("Int8")

    # Preserve the intended field order before writing.
    chunk = chunk[TABLEAU_COLUMNS]

    # Write the header only for the first chunk, then append later chunks.
    chunk.to_csv(
        TEMP_OUTPUT_FILE,
        mode="w" if chunk_number == 1 else "a",
        header=chunk_number == 1,
        index=False,
        encoding="utf-8",
        na_rep="",
    )

    total_rows_written += len(chunk)
    print(
        f"Processed chunk {chunk_number}: "
        f"{total_rows_written:,} total rows"
    )

# Replace an older completed output only after the new file finishes writing.
TEMP_OUTPUT_FILE.replace(OUTPUT_FILE)

print("\nTableau export completed.")
print(f"Rows written: {total_rows_written:,}")
print(f"Invalid received dates: {invalid_received_dates:,}")
print(f"Invalid month-start dates: {invalid_month_dates:,}")
print(f"Output size: {OUTPUT_FILE.stat().st_size / (1024 ** 2):,.1f} MB")
print(f"Saved to: {OUTPUT_FILE}")

## Validate the exported file

The final file should contain 888,691 rows and 888,691 distinct complaint IDs, covering January 1, 2021 through December 31, 2025. Product totals should also match the cleaned MySQL table.

In [4]:
EXPECTED_ROWS = 888_691

# Load only the few columns needed for validation.
validation = pd.read_csv(
    OUTPUT_FILE,
    usecols=[
        "complaint_id",
        "received_date",
        "received_year",
        "product_group",
    ],
    parse_dates=["received_date"],
    low_memory=False,
)

total_rows = len(validation)
distinct_ids = validation["complaint_id"].nunique(dropna=True)
missing_ids = int(validation["complaint_id"].isna().sum())
earliest_date = validation["received_date"].min().date()
latest_date = validation["received_date"].max().date()

print(f"Total rows: {total_rows:,}")
print(f"Distinct complaint IDs: {distinct_ids:,}")
print(f"Missing complaint IDs: {missing_ids:,}")
print(f"Date range: {earliest_date} through {latest_date}")

print("\nComplaint totals by product:")
display(
    validation["product_group"]
    .value_counts()
    .rename_axis("product_group")
    .reset_index(name="complaint_count")
)

# Stop with a clear error if the Tableau export does not match the clean table.
assert total_rows == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS:,} rows but found {total_rows:,}."
)
assert distinct_ids == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS:,} distinct complaint IDs but found "
    f"{distinct_ids:,}."
)
assert missing_ids == 0, "The Tableau export contains missing complaint IDs."
assert str(earliest_date) == "2021-01-01"
assert str(latest_date) == "2025-12-31"

print("\nValidation successful: the Tableau file matches the clean dataset.")

Total rows: 888,691
Distinct complaint IDs: 888,691
Missing complaint IDs: 0
Date range: 2021-01-01 through 2025-12-31

Complaint totals by product:


,product_group,complaint_count
0,Cards,309937
1,Checking and Savings,254027
2,Money Transfers and Digital Payments,142549
3,Mortgage,118793
4,Vehicle Loans and Leases,63385



Validation successful: the Tableau file matches the clean dataset.


## Preview the Tableau-ready data

In [5]:
tableau_preview = pd.read_csv(
    OUTPUT_FILE,
    nrows=10,
    parse_dates=["received_date", "month_start_date"],
)

display(tableau_preview)
print("\nColumn data types in the preview:")
print(tableau_preview.dtypes)

,complaint_id,received_date,received_year,month_start_date,product_group,issue,sub_issue,company,state,tags,submitted_via,company_response,is_timely
0,4041896,2021-01-01,2021,2021-01-01,Cards,Trouble using the card,Trouble getting information about the card,AMERICAN EXPRESS COMPANY,OH,NaN,Web,Closed with monetary relief,1
1,4041897,2021-01-01,2021,2021-01-01,Mortgage,Struggling to pay mortgage,NaN,Mr. Cooper Group Inc.,TX,Servicemember,Web,Closed with explanation,1
2,4041911,2021-01-01,2021,2021-01-01,Checking and Savings,Closing an account,Funds not received from closed account,"BANK OF AMERICA, NATIONAL ASSOCIATION",IL,NaN,Web,Closed with monetary relief,1
3,4041913,2021-01-01,2021,2021-01-01,Checking and Savings,Managing an account,Problem using a debit or ATM card,JPMORGAN CHASE & CO.,TX,NaN,Web,Closed with explanation,1
4,4041915,2021-01-01,2021,2021-01-01,Money Transfers and Digital Payments,"Overdraft, savings, or rewards features",NaN,Google Compare Credit Cards Inc.,CA,NaN,Web,Closed with explanation,1
5,4041916,2021-01-01,2021,2021-01-01,Checking and Savings,Closing an account,Funds not received from closed account,"CITIBANK, N.A.",IL,NaN,Web,Closed with explanation,1
6,4041917,2021-01-01,2021,2021-01-01,Mortgage,Trouble during payment process,NaN,"MONEY SOURCE, INC., THE",CA,NaN,Web,Closed with explanation,1
7,4041930,2021-01-01,2021,2021-01-01,Checking and Savings,Managing an account,Banking errors,"BANK OF AMERICA, NATIONAL ASSOCIATION",CA,Servicemember,Web,Closed with explanation,1
8,4041933,2021-01-01,2021,2021-01-01,Mortgage,Trouble during payment process,NaN,Mr. Cooper Group Inc.,AP,NaN,Web,Closed with explanation,1
9,4041935,2021-01-01,2021,2021-01-01,Mortgage,Applying for a mortgage or refinancing an exis...,NaN,"Summerfield Mortgage, LLC",GA,NaN,Web,Closed with explanation,1



Column data types in the preview:
complaint_id                 int64
received_date       datetime64[us]
received_year                int64
month_start_date    datetime64[us]
product_group                  str
issue                          str
sub_issue                      str
company                        str
state                          str
tags                           str
submitted_via                  str
company_response               str
is_timely                    int64
dtype: object
